### 1. The MDP Model

Consider once again the garbage collection problem described in the Homework and for which you partially wrote a Markov decision problem model. In this lab, you will consider a larger version and slightly modified version of the same problem, described by the diagram:
# 
# <img src="garbage-big.png">
# 
Recall that the MDP should describe the decision-making process of the truck driver. In the above domain, 

* At any time step, garbage is _at most_ in one of the cells marked with a garbage bin. 
* When the garbage truck picks up the garbage from one of the bins, it becomes ``loaded``. 
* While the truck is loaded, no garbage appears in any of the marked locations.
* The driver has six actions available: `Up`, `Down`, `Left`, `Right`, `Pick`, and `Drop`. 
* Each movement action moves the truck to the adjacent stop in the corresponding direction, if there is one. Otherwise, it has no effect. 
* The `Pick` action succeeds when the truck is in a location with garbage. In that case, the truck becomes "loaded".
* The `Drop` action succeeds when the loaded truck is at the recycling plant. After a successful drop, the truck becomes empty, and garbage may now appear in any of the marked cells with a total probability of 0.3.
 
In this lab you will use an MDP based on the aforementioned domain and investigate how to evaluate, solve and simulate a Markov decision problem.
 
**Throughout the lab, unless if stated otherwise, use $\gamma=0.99$.**
 
$$\diamond$$
 
In this first activity, you will implement an MDP model in Python. You will start by loading the MDP information from a `numpy` binary file, using the `numpy` function `load`. The file contains the list of states, actions, the transition probability matrices and cost function. After you load the file, you can index the resulting object as a dictionary to access the different elements.


#### Activity 1.        
 
Write a function named `load_mdp` that receives, as input, a string corresponding to the name of the file with the MDP information, and a real number $\gamma$ between $0$ and $1$. The loaded file will contain 4 arrays:
 
* An array `X` that contains all the states in the MDP represented as strings. In the garbage collection environment above, for example, there is a total of 462 states, each describing the location of the truck in the environment, the location of the garbage (or `None` if no garbage exists in the environment), and whether the truck is `loaded` or `empty`. Each state is, therefore, a string of the form `"(p, g, t)"`, where:
* `p` is one of `0`, ..., `32`, indicating the location of the truck;
* `g` is either `None` or one of `1`, `9`, `10`, `11`, `18`, `19`, `20`, `21`, `23`, `27`, `28`, `29`, indicating that no garbage exists (`None`), or that there is garbage in one of the listed stops;     * `t` is either `empty` or `loaded`, indicating whether the truck is loaded or not.
* An array `A` that contains all the actions in the MDP, represented as strings. In the garbage collection environment above, for example, each action is represented as a string `"Up"`, `"Down"`, `"Left"`, `"Right"`, `"Pick"`, and `"Drop"`.
* An array `P` containing `len(A)` subarrays, each with dimension `len(X)` &times; `len(X)` and  corresponding to the transition probability matrix for one action.
* An array `c` with dimension `len(X)` &times; `len(A)` containing the cost function for the MDP.

Your function should create the MDP as a tuple `(X, A, (Pa, a = 0, ..., len(A)), c, g)`, where `X` is a tuple containing the states in the MDP represented as strings (see above), `A` is a tuple containing the actions in the MDP represented as strings (see above), `P` is a tuple with `len(A)` elements, where `P[a]` is an `np.array` corresponding to the transition probability matrix for action `a`, `c` is an `np.array` corresponding to the cost function for the MDP, and `g` is a float, corresponding to the discount and provided as the argument $\gamma$ of your function. Your function should return the MDP tuple.

In [3]:
import numpy as np

In [4]:
def load_mdp(file: str, gamma: float) -> tuple:
    input = np.load(file)
    states = tuple()
    n_states = input["X"].shape[0]
    for i in range(n_states):
        states += (input["X"][i], )
    actions = tuple()
    n_actions = input["A"].shape[0]
    for i in range(n_states):
        states += (input["A"][i], )
    transition_probabilities = tuple()
    for i in range(input["P"].shape[0]):
        transition_probabilities += (input["P"][i], )
    cost_function = input["c"]
    return (states, actions, cost_function, gamma)
    

We provide below an example of application of the function with the file `garbage-big.npz` that you can use as a first "sanity check" for your code. Note that, even fixing the seed, the results you obtain may slightly differ.

```python
import numpy.random as rand

M = load_mdp('garbage-big.npz', 0.99)

rand.seed(42)

# States
print('= State space (%i states) =' % len(M[0]))
print('\nStates:')
for i in range(min(10, len(M[0]))):
    print(M[0][i]) 

print('...')

# Random state
s = rand.randint(len(M[0]))
print('\nRandom state: s =', M[0][s])

# Last state
print('Last state: s =', M[0][-1])

# Actions
print('\n= Action space (%i actions) =\n' % len(M[1]))
for i in range(len(M[1])):
    print(M[1][i]) 

# Random action
a = rand.randint(len(M[1]))
print('\nRandom action: a =', M[1][a])

# Transition probabilities
print('\n= Transition probabilities =')

for i in range(len(M[1])):
    print('\nTransition probability matrix dimensions (action %s):' % M[1][i], M[2][i].shape)
    print('Dimensions add up for action "%s"?' % M[1][i], np.isclose(np.sum(M[2][i]), len(M[0])))
    
print('\nState-action pair (%s, %s) transitions to state(s)' % (M[0][s], M[1][a]))
print("s' in", np.array(M[0])[np.where(M[2][a][s, :] > 0)])

# Cost
print('\n= Costs =')
print('\nCost for the state-action pair (%s, %s):' % (M[0][s], M[1][a]))
print('c(s, a) =', M[3][s, a])


# Discount
print('\n= Discount =')
print('\ngamma =', M[4])
```

Output:

```
= State space (462 states) =

States:
(0, None, empty)
(0, 1, empty)
(0, 9, empty)
(0, 10, empty)
(0, 11, empty)
(0, 18, empty)
(0, 19, empty)
(0, 20, empty)
(0, 21, empty)
(0, 23, empty)
...

Random state: s = (7, 28, empty)
Last state: s = (32, None, loaded)

= Action space (6 actions) =

Up
Down
Left
Right
Pick
Drop

Random action: a = Right

= Transition probabilities =

Transition probability matrix dimensions (action Up): (462, 462)
Dimensions add up for action "Up"? True

Transition probability matrix dimensions (action Down): (462, 462)
Dimensions add up for action "Down"? True

Transition probability matrix dimensions (action Left): (462, 462)
Dimensions add up for action "Left"? True

Transition probability matrix dimensions (action Right): (462, 462)
Dimensions add up for action "Right"? True

Transition probability matrix dimensions (action Pick): (462, 462)
Dimensions add up for action "Pick"? True

Transition probability matrix dimensions (action Drop): (462, 462)
Dimensions add up for action "Drop"? True

State-action pair ((7, 28, empty), Right) transitions to state(s)
s' in ['(8, 28, empty)']

= Costs =

Cost for the state-action pair ((7, 28, empty), Right):
c(s, a) = 0.501

= Discount =

gamma = 0.99
```

**Note:** For debug purposes, we also provide a second file, `garbage-small.npz`, that contains a 6-state MDP that you can use to verify if your results make sense.

### 2. Prediction

You are now going to evaluate a given policy, computing the corresponding cost-to-go.

---

#### Activity 2.

Write a function `noisy_policy` that builds a noisy policy "around" a provided action. Your function should receive, as input, an MDP described as a tuple like that of **Activity 1**, an integer `a`, corresponding to the index of an action in the MDP, and a real number `eps`. The function should return, as output, a policy for the provided MDP that selects action with index `a` with a probability `1 - eps` and, with probability `eps`, selects another action uniformly at random. The policy should be a `numpy` array with as many rows as states and as many columns as actions, where the element in position `[s,a]` should contain the probability of action `a` in state `s` according to the desired policy.

**Note:** The examples provided correspond for the MDP in the previous garbage collection environment. However, your code should be tested with MDPs of different sizes, so **make sure not to hard-code any of the MDP elements into your code**.

---

In [5]:
def noisy_policy(MDP: tuple, a: int, eps: float) -> np.array:
    policy = np.zeros((len(MDP[0]), len(MDP[1]))) # zero matrix with #n_states x #n_actions
    n_actions = len(MDP[1])

    for i in range(policy.shape[0]): #state
        for j in range(policy.shape[1]): #action
            if j == a:
                policy[i, j] = 1 - eps
            else:
                policy[i, j] = eps / (n_actions-1)
    
    return policy

---

#### Activity 3.

You will now write a function called `evaluate_pol` that evaluates a given policy. Your function should receive, as an input, an MDP described as a tuple like that of **Activity 1** and a policy described as an array like that of **Activity 2** and return a `numpy` array corresponding to the cost-to-go function associated with the given policy. 

**Note:** The array returned by your function should have as many rows as the number of states in the received MDP, and exactly one column. Note also that, as before, your function should work with **any** MDP that is specified as a tuple with the same structure as the one from **Activity 1**. In your solution, you may find useful the function `np.linalg.inv`, which can be used to invert a matrix.

---

In [6]:
def evaluate_pol(MDP: tuple, policy: np.array) -> np.array:
    # J ^ p = (I - gamma * policy) c_p
    P_policy = np.zeros((len(MDP[0], len(MDP[1]))))
    for i in range(len(MDP[1])): #FIXME nao sei bem como se faz, esta igual ao ano passado
        P_policy += policy[:, [i]] * MDP[2][i]
    c_policy = np.sum(policy * MDP[3], axis=1)
    I = np.eye(P_policy.shape[0])
    aux = np.dot(MDP[4], P_policy)
    return np.dot(np.linalg.inv(I - aux), c_policy)

### 3. Control

In this section you are going to compare value and policy iteration, both in terms of time and number of iterations.

---

#### Activity 4

In this activity you will show that the policy in Activity 3 is _not_ optimal. For that purpose, you will use value iteration to compute the optimal cost-to-go, $J^*$, and show that $J^*\neq J^\pi$. 

Write a function called `value_iteration` that receives as input an MDP represented as a tuple like that of **Activity 1** and returns an `numpy` array corresponding to the optimal cost-to-go function associated with that MDP. Before returning, your function should print:

* The time it took to run, in the format `Execution time: xxx seconds`, where `xxx` represents the number of seconds rounded up to $3$ decimal places.
* The number of iterations, in the format `N. iterations: xxx`, where `xxx` represents the number of iterations.

**Note 1:** Stop the algorithm when the error between iterations is smaller than $10^{-8}$. To compute the error between iterations, you should use the function `norm` from `numpy.linalg`. 

**Note 2:** You may find useful the function ``time()`` from the module ``time``. You may also find useful the code provided in the theoretical lecture.

**Note 3:** The array returned by your function should have as many rows as the number of states in the received MDP, and exactly one column. As before, your function should work with **any** MDP that is specified as a tuple with the same structure as the one from **Activity 1**.


---

In [7]:
import time

def value_iteration(MDP: tuple) -> np.array:
    # lecture 05 slide 92
    t0 = time()

    X = MDP[0]
    A = MDP[1]
    P = MDP[2]
    c = MDP[3]
    gamma = MDP[4]

    J = np.zeros((len(X), 1))
    error = np.float32(1e-8)
    print(error)

    iterations = 0
    while True:
        Q = np.zeros((len(X), len(A)))

        for a in range(len(A)):
            Q[:, a, None] = c[:, a, None] + gamma * P[a].dot(J)

        Jnew = np.min(Q, axis=1, keepdims=True)

        error = np.linalg.norm(J - Jnew)

        J = Jnew
        iterations += 1

        if error > np.float32(1e-8):
            break

    print(f"Execution time: {round(t0 - time(), 3)}")
    print(f"N. iterations: {iterations}")
    return J[:, None]

---

#### Activity 5

You will now compute the optimal policy using policy iteration. Write a function called `policy_iteration` that receives as input an MDP represented as a tuple like that of **Activity 1** and returns an `numpy` array corresponding to the optimal policy associated with that MDP. Before returning, your function should print:
* The time it took to run, in the format `Execution time: xxx seconds`, where `xxx` represents the number of seconds rounded up to $3$ decimal places.
* The number of iterations, in the format `N. iterations: xxx`, where `xxx` represents the number of iterations.

**Note:** If you find that numerical errors affect your computations (especially when comparing two values/arrays) you may use the `numpy` function `isclose` with adequately set absolute and relative tolerance parameters (e.g., $10^{-8}$). You may also find useful the code provided in the theoretical lecture.

---

In [ ]:
def policy_iteration(MDP: tuple)